<a href="https://colab.research.google.com/github/pmadhyastha/INM434/blob/main/from_tranformers_to_BERT_to_GPT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
__author__ = "Pranava Madhyastha"
__version__ = "INM434/IN3045 City, University of London, Spring 2025"

# Attention and Transformers

We are repeating transformers from the last lecture! Please focus on the basic elements so that you are able to move on to more sophisticated algorithms with transformers.


### TODO: Please print out the values and verify if you are able to understand each step below. Use lecture slides as the reference.

In [2]:
import torch
import torch.nn.functional as F

# The sample sentence

We will begin with a sample sentence that we have seen in the lectures: `the cat is on a new mat`


In [3]:
# Define the input sentence
input_sentence = 'the cat is on a new mat'

# Creation of the dictionary

The next step is to create a dictionary that maps each word in the input sentence to a unique index. This is done by splitting the sentence into a list of words.

The list of words is then sorted, and each word is assigned an index using the enumerate() function. The result is a dictionary that maps each word to a unique index.

In [4]:
# Create a dictionary that maps each word to a unique index
word_to_index = {word: i for i, word in enumerate(sorted(input_sentence.split()))}

# Tensor of indices

The input sentence is then converted to a tensor of indices using the word_to_index dictionary. The tensor is created by iterating over each word in the input sentence (after removing the comma) and looking up its index in the dictionary. The resulting tensor is a one-dimensional tensor of integer indices.

In [5]:
# Convert the input sentence to a tensor of indices using the word_to_index dictionary
input_tensor = torch.tensor([word_to_index[word] for word in input_sentence.replace(',', '').split()])


# The embedding layer

The next step is to define an embedding layer. The embedding layer is a PyTorch module that learns a lookup table of word embeddings for a vocabulary of size n and embedding dimensionality d. In this case, the embedding layer is initialized with a vocabulary size equal to the number of words in the input sentence and an embedding dimensionality of 16. The input tensor is then embedded using the embedding layer to produce an embedded sentence. The .detach() method is used to detach the embedded sentence from its computation graph, preventing it from being backpropagated through.

In [6]:
# Define the embedding layer
embedding_layer = torch.nn.Embedding(len(word_to_index), 16)

# Embed the input tensor to get the embedded sentence
embedded_sentence = embedding_layer(input_tensor).detach()

The size of the embedding is then defined by taking the shape of the embedded sentence and extracting its second dimension.

In [7]:
# Define the size of the embedding
embedding_size = embedded_sentence.shape[1]

# Query, key and value

The sizes of the query, key, and value vectors are defined. Recall that these vectors are used in the attention mechanism to compute the attention weights and context vectors. In this case, the query and key vectors have a size of 24, while the value vector has a size of 28.

In [8]:
# Define the sizes of the query, key, and value vectors
query_size, key_size, value_size = 24, 24, 28


# Initialisation and projection

Query, key, and value weight matrices are then defined using PyTorch's random number generator. The weight matrices are used to compute the query, key, and value vectors for each word in the input sentence. Refer to the lecture slides!

In [9]:
# Define the query, key, and value weight matrices
query_weights = torch.rand(query_size, embedding_size)
key_weights = torch.rand(key_size, embedding_size)
value_weights = torch.rand(value_size, embedding_size)

# Example computation

The query, key, and value vectors for the second word in the input sentence are then computed by multiplying the corresponding weight matrices by the embedding vector for the second word.

In [10]:
# Get the query, key, and value vectors for the second word in the input sentence
x_2 = embedded_sentence[1]
query_2 = query_weights.matmul(x_2)
key_2 = key_weights.matmul(x_2)
value_2 = value_weights.matmul(x_2)

# Parallelisation

The keys and values for all words in the input sentence are then computed by multiplying the corresponding weight matrices by the transpose of the embedded sentence. The transpose operation is used to swap the rows and columns of the embedded sentence, making it possible to compute the dot product between the weight matrices and the embedding vectors. The transpose operation is then applied again to swap the rows and columns back to their original order.

In [11]:
# Compute the keys and values for all words in the input sentence
keys = key_weights.matmul(embedded_sentence.transpose(0, 1)).transpose(0, 1)
values = value_weights.matmul(embedded_sentence.transpose(0, 1)).transpose(0, 1)

# Computing the attention weights

The attention weights for the second word in the input sentence are then computed using the softmax function applied to the dot product of the query vector for the second word and the transpose of the key vectors for all words in the input sentence. The softmax function normalizes the dot product, resulting in a set of attention weights that sum to one. The size of the key vectors is used to scale the dot product.



In [12]:
# Compute the keys and values for all words in the input sentence
keys = key_weights.matmul(embedded_sentence.transpose(0, 1)).transpose(0, 1)
values = value_weights.matmul(embedded_sentence.transpose(0, 1)).transpose(0, 1)

# Computing `h` or the context vector

The context vector for the second word in the input sentence is then computed by multiplying the attention weights by the value vectors for all words in the input sentence. The result is a weighted sum of the value vectors, where the weights are given by the attention weights.

In [13]:
# Compute the attention weights for the second word in the input sentence
omega_2 = query_2.matmul(keys.T)
attention_weights_2 = F.softmax(omega_2 / key_size **0.5, dim=0)


# Compute the context vector for the second word in the input sentence
context_vector_2 = attention_weights_2.matmul(values)

# Multi-head attention

The code then defines the number of attention heads to use, which is set to 3. It also defines new query, key, and value weight matrices for the multi-head attention mechanism. The size of each weight matrix is expanded to include a third dimension for the number of attention heads.

Refer to the lecture notes to understand the concept of Multi-head attention.

In [14]:
# Define the number of attention heads
num_heads = 3

# Define the query, key, and value weight matrices for the multi-head attention
multihead_query_weights = torch.rand(num_heads, query_size, embedding_size)
multihead_key_weights = torch.rand(num_heads, key_size, embedding_size)
multihead_value_weights = torch.rand(num_heads, value_size, embedding_size)


The query vector for the second word in the input sentence is obtained by multiplying the embedded vector of the second word with the multi-head query weight matrix. This is done separately for each attention head.

In [15]:
# Get the query vector for the second word in the input sentence for each attention head
multihead_query_2 = multihead_query_weights.matmul(x_2)

# Repeat

The embedded sentence is repeated for each attention head using the repeat function. This allows each head to have its own set of keys and values for computing attention.

In [16]:
# Repeat the embedded sentence for each attention head
repeated_embedded_sentence = embedded_sentence.unsqueeze(0).repeat(num_heads, 1, 1)

The keys and values for all words in the input sentence are computed for each attention head by multiplying the repeated embedded sentence with the multi-head key and value weight matrices. The transpose function is used to reshape the tensor dimensions so that the matrix multiplication can be computed.

In [17]:
# Compute the keys and values for all words in the input sentence for each attention head
multihead_keys = multihead_key_weights.matmul(repeated_embedded_sentence.transpose(1, 2)).transpose(1, 2)
multihead_values = multihead_value_weights.matmul(repeated_embedded_sentence.transpose(1, 2)).transpose(1, 2)

# Compute the attention weights and context vectors for each attention head
attention_weights = F.softmax(multihead_query_2.matmul(multihead_keys.transpose(1, 2)) / key_size**0.5, dim=2)
context_vectors = attention_weights.matmul(multihead_values)


### TODO: Please follow the annotated transformer: http://nlp.seas.harvard.edu/annotated-transformer/ and consider re-writing and running it here on google colab.



# BERT based classifier

In the following sections, we are going to work on the BERT classifier for sentiment analysis - trained and tested over the IMDB dataset.

In [18]:
!pip install transformers
!pip install datasets # remember lab3?

The above code gets us pre-optimised versions (and pre-trained weights) for a BERT based model. We are also downloading the datasets library - we had also done this before.

In [19]:
import torch
import torch.nn as nn
from torch.optim import AdamW
from transformers import DistilBertModel, DistilBertTokenizer
from datasets import load_dataset

# Load the dataset

dataset = load_dataset('imdb')

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

plain_text/train-00000-of-00001.parquet:   0%|          | 0.00/21.0M [00:00<?, ?B/s]

plain_text/test-00000-of-00001.parquet:   0%|          | 0.00/20.5M [00:00<?, ?B/s]

plain_text/unsupervised-00000-of-00001.p(…):   0%|          | 0.00/42.0M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating unsupervised split:   0%|          | 0/50000 [00:00<?, ? examples/s]

We will not work with BERT directly (as it may take a while to run), instead we shall use a "distilled version" of BERT (reduced model size) called [DistillBERT](https://arxiv.org/abs/1910.01108) . The code below loads the IMDB dataset from Hugging Face and preprocesses it. It loads the tokenizer and model from the transformers library, tokenizes the text using the DistilBERT tokenizer, and converts the data to the PyTorch format.

In [20]:
# Load the tokenizer and model
tokenizer = DistilBertTokenizer.from_pretrained('distilbert-base-uncased')
base_model = DistilBertModel.from_pretrained('distilbert-base-uncased')


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


The code below just defines the architecture the model architecture using a PyTorch neural network module. The SentimentClassifier class takes the DistilBERT model as input and defines the forward method. It extracts the last hidden state from the output of the DistilBERT model, and passes it through a linear layer to get the final logits.

In [21]:
# Define the model architecture
class SentimentClassifier(nn.Module):
    def __init__(self, model):
        super(SentimentClassifier, self).__init__()
        self.model = model
        self.linear = nn.Linear(768, 1)

    def forward(self, input_ids, attention_mask):
        outputs = self.model(input_ids=input_ids, attention_mask=attention_mask)
        last_hidden_state = outputs.last_hidden_state[:, 0, :]
        logits = self.linear(last_hidden_state)
        return logits.squeeze(-1)

# Instantiate the model
model = SentimentClassifier(base_model)



The code below now defines the training and eval loop. It sets the model to training mode and iterates through each batch of the training data. It calculates the loss, computes the gradients, and updates the model parameters using the AdamW optimizer.

In [22]:
# Tokenize the data
def tokenize(batch):
    return tokenizer(batch['text'], padding=True, truncation=True)

train_dataset = dataset['train'].map(tokenize, batched=True)
test_dataset = dataset['test'].map(tokenize, batched=True)

# Set the data format
train_dataset.set_format('torch', columns=['input_ids', 'attention_mask', 'label'])
test_dataset.set_format('torch', columns=['input_ids', 'attention_mask', 'label'])

# Set up the optimizer and loss function
optimizer = AdamW(model.parameters(), lr=5e-5)
loss_fn = nn.BCEWithLogitsLoss()

# Define the training loop
def train(model, train_loader, optimizer, loss_fn, device):
    model.train()
    for batch in train_loader:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['label'].to(device)

        optimizer.zero_grad()
        outputs = model(input_ids, attention_mask)
        loss = loss_fn(outputs, labels.float())
        print(loss)
        loss.backward()
        optimizer.step()

# Define the evaluation loop
def evaluate(model, test_loader, device):
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for batch in test_loader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['label'].to(device)

            outputs = model(input_ids, attention_mask)
            predictions = torch.round(torch.sigmoid(outputs))

            total += labels.size(0)
            correct += (predictions == labels).sum().item()

    return correct / total




Map:   0%|          | 0/25000 [00:00<?, ? examples/s]

Map:   0%|          | 0/25000 [00:00<?, ? examples/s]

We will now formally train the model.

In [23]:
# Train the model
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)

train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=32, shuffle=False)

for epoch in range(1):
    train(model, train_loader, optimizer, loss_fn, device)
    accuracy = evaluate(model, test_loader, device)
    print(f'Epoch {epoch+1} - Test Accuracy: {accuracy:.3f}')

tensor(0.6661, device='cuda:0',
       grad_fn=<BinaryCrossEntropyWithLogitsBackward0>)
tensor(0.7659, device='cuda:0',
       grad_fn=<BinaryCrossEntropyWithLogitsBackward0>)
tensor(0.7131, device='cuda:0',
       grad_fn=<BinaryCrossEntropyWithLogitsBackward0>)
tensor(0.6870, device='cuda:0',
       grad_fn=<BinaryCrossEntropyWithLogitsBackward0>)
tensor(0.7112, device='cuda:0',
       grad_fn=<BinaryCrossEntropyWithLogitsBackward0>)
tensor(0.6301, device='cuda:0',
       grad_fn=<BinaryCrossEntropyWithLogitsBackward0>)
tensor(0.6145, device='cuda:0',
       grad_fn=<BinaryCrossEntropyWithLogitsBackward0>)
tensor(0.6364, device='cuda:0',
       grad_fn=<BinaryCrossEntropyWithLogitsBackward0>)
tensor(0.6289, device='cuda:0',
       grad_fn=<BinaryCrossEntropyWithLogitsBackward0>)
tensor(0.5895, device='cuda:0',
       grad_fn=<BinaryCrossEntropyWithLogitsBackward0>)
tensor(0.5532, device='cuda:0',
       grad_fn=<BinaryCrossEntropyWithLogitsBackward0>)
tensor(0.5127, device='cuda:0',


KeyboardInterrupt: 

# Todo:
- Test it on other sentiment datasets, does it generalise across?
- How good is the generalisation?
- Have a look at huggingface library and replace distillBERT with the BERT-base-uncased version, rerun the model. Compare the results.




In [ ]:
!pip uninstall -y wandb  # Explicitly uninstall wandb
import os
os.environ["WANDB_DISABLED"] = "true" # Set environment variable to disable wandb
os.environ["WANDB_MODE"] = "disabled"
!echo "WandB disabled forcefully."

# We will cover the following topics:

# 1. **Loading and Using a Pre-trained LLM:** We'll start by loading the pre-trained GPT-2 model and use it for text generation.
# 2. **Fine-tuning an LLM for Sentiment Analysis:** We will then fine-tune GPT-2 for a specific task - sentiment analysis of movie reviews.
# 3. **Direct Preference Optimization (DPO):** Finally, we'll delve into a simplified Reinforcement Learning from Human Feedback (RLHF) technique called Direct Preference Optimization to align our model with preferences, using a summarization dataset.

# **Before you begin:**

# *   **Runtime Environment:** Make sure you are running this notebook in an environment with **GPU acceleration enabled**. In Google Colab, you can do this by going to "Runtime" -> "Change runtime type" and selecting "A100 GPU" as the hardware accelerator.


In [ ]:
!pip install -U fsspec
!pip install -q -U transformers datasets accelerate bitsandbytes peft trl

# Let's start by loading a pre-trained **GPT-2** model and using it for text generation. We will use the `pipeline` from Hugging Face `transformers` library, which simplifies the process of using pre-trained models for various NLP tasks.

# **Model Hub - Hugging Face:**

# Before we load the model, let's briefly talk about the **Hugging Face Hub** ([https://huggingface.co/models](https://huggingface.co/models)). This is a central repository where thousands of pre-trained models, datasets, and other resources are shared by the community. You can find models for various tasks and languages here.

# **Finding GPT-2 Models:** You can find GPT-2 models by searching for "GPT-2" on the Hugging Face Hub. Open AI community hosts official GPT-2 models under the organization "openai-community".

# We will be using the **"distillgpt2"** as an example.

In [ ]:
!pip install -U huggingface_hub
from huggingface_hub import login
login()

# What does the model generate?

# Examine the generated stories. Are they coherent? Do they follow the prompt?
# GPT-2, being a powerful LLM, should generate reasonably good and creative stories based on the prompt.

# The `pipeline` handles all the complexities behind the scenes, including:
# *   **Downloading the model weights:** It automatically downloads the model weights from the Hugging Face Hub if they are not already cached locally.
# *   **Loading the tokenizer:** It loads the appropriate tokenizer associated with the GPT-2 model.
# *   **Performing text generation:** It uses the model and tokenizer to generate text based on your prompt.



# 2. Fine-tuning GPT-2 for Sentiment Analysis

# Pre-trained LLMs are versatile, but fine-tuning them on a specific dataset can significantly improve their performance on a particular task. Let's fine-tune GPT-2 for **sentiment analysis**. We'll use the **IMDB reviews dataset**, a standard dataset for sentiment classification.


In [ ]:
from datasets import load_dataset

# Load the IMDB sentiment analysis dataset -- we have seen this before, haven't we?
dataset = load_dataset("imdb")
print(dataset) # check the dataset please -- does this make sense?

The IMDB dataset is divided into 'train' and 'test' splits. Each example contains:
*   `text`: The movie review text.
*   `label`: The sentiment label (0 for negative, 1 for positive).

Now, we need to tokenize the text data so that it can be processed by the GPT-2 model. We'll use the tokenizer associated with GPT-2

from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(model_name) #
tokenizer.pad_token = tokenizer.eos_token # Important: Set pad token to EOS token for GPT-2 models -- we are trying a hack here!

def tokenize_function(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True, max_length=128) # Tokenize and pad/truncate sequences

tokenized_datasets = dataset.map(tokenize_function, batched=True) # Apply tokenization to the entire dataset
print(tokenized_datasets) # Inspect the tokenized dataset now

In this step, we used the `AutoTokenizer` to load the tokenizer for GPT-2.  `AutoTokenizer` automatically fetches the correct tokenizer configuration from the model repository on Hugging Face Hub.

We then defined a `tokenize_function` that:
 *   Takes text examples as input.
 *   Tokenizes the text using the GPT-2 tokenizer.
 *   Applies `padding="max_length"` to pad sequences to a maximum length (128 in this case).
 *   Applies `truncation=True` to truncate sequences longer than the maximum length.

The `dataset.map()` function applies this `tokenize_function` to the entire IMDB dataset in batches, resulting in `tokenized_datasets`.

Now, we will set up the training pipeline using Hugging Face `Trainer` to fine-tune GPT-2 for sentiment classification.


In [ ]:
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer

# please have a look at AutoModelForSequenceClassification -- this is a specialised module for prediction over a few classes!

# Load distilgpt2 for sequence classification with 2 output labels (positive/negative)
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)
model.config.pad_token_id = model.config.eos_token_id

training_args = TrainingArguments(
    output_dir="./distilgpt2-sentiment-model",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=2,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_steps=100,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    push_to_hub=False,
    report_to="none",  # Disable WandB logging (again, for good measure)
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["test"], #change this
    eval_dataset=tokenized_datasets["test"],
    processing_class=tokenizer,
    # We can add metrics computation here if needed, but for simplicity we'll focus on loss
)

trainer.train() # Start the fine-tuning process

The `trainer.train()` command starts the fine-tuning process. This will take some time depending on your GPU and the number of epochs. You will see training logs printed during the process, including loss values and evaluation results at the end of each epoch.

# **Important Note:** Training large models like GPT-2 can be computationally intensive. If you are running into memory issues (crashes or out-of-memory errors), try reducing the `per_device_train_batch_size`, `per_device_eval_batch_size`, or `max_length` in the `tokenize_function`. You can also reduce `num_train_epochs` for faster experimentation, but keep in mind that fewer epochs might lead to less optimal performance.

After training is complete, the best model (based on `eval_loss`) will be loaded. Let's evaluate the fine-tuned model.

In [ ]:
evaluation_results = trainer.evaluate()
print(evaluation_results)

The `trainer.evaluate()` function calculates the loss on the evaluation dataset (the 'test' split of IMDB).  The `eval_loss` value indicates how well the model is performing on unseen data. A lower `eval_loss` generally means better performance.

In [ ]:
def predict_sentiment(sentence):
    inputs = tokenizer(sentence, return_tensors="pt", truncation=True, padding=True).to(model.device) # Tokenize and move to device
    outputs = model(**inputs)                                                                    # Get model outputs
    predictions = outputs.logits.argmax(dim=-1)                                                     # Get predicted class (0 or 1)
    sentiment = "positive" if predictions.item() == 1 else "negative"
    return f"Sentiment: {sentiment}"

sentences = [
    "This movie was amazing! The acting was superb and the plot was captivating.",
    "I absolutely hated this film. It was boring and predictable.",
    "It was a mediocre movie, nothing special but not terrible."
]

print("\nSentiment Predictions from Fine-tuned GPT-2:")
for sentence in sentences:
    print(f"- Sentence: '{sentence}' - {predict_sentiment(sentence)}")

Observe the sentiment predictions for the example sentences. Does the fine-tuned GPT-2 model correctly classify the sentiment?  Try experimenting with your own sentences to see how well it performs!

You have now successfully fine-tuned GPT-2 for sentiment analysis! This demonstrates the power of fine-tuning pre-trained LLMs for specific downstream tasks.


In [ ]:
from trl import DPOTrainer
from transformers import AutoModelForCausalLM

Now, let's explore a more advanced technique called **Direct Preference Optimization (DPO)**. DPO is a simplified approach to **Reinforcement Learning from Human Feedback (RLHF)**. RLHF aims to align LLMs with human preferences, making them generate outputs that humans find more desirable.

Traditional RLHF methods can be complex, involving training a reward model and using algorithms like Proximal Policy Optimization (PPO). DPO simplifies this by directly optimizing the language model based on pairwise preference data, without explicitly training a separate reward model.

In DPO, we use datasets where for a given prompt, there are two model outputs: a "chosen" output and a "rejected" output. The "chosen" output is preferred over the "rejected" one. DPO learns to increase the likelihood of the chosen outputs and decrease the likelihood of rejected outputs relative to the chosen ones.

We'll use the `trl` (Transformer Reinforcement Learning) library from Hugging Face to demonstrate DPO. We'll use the **"CarperAI/openai_summarize_comparisons"** dataset, which contains summarization preferences.


In [ ]:
dpo_dataset = load_dataset("CarperAI/openai_summarize_comparisons", split="train")
print(dpo_dataset[0]) # Inspect an example from the DPO dataset

 The "CarperAI/openai_summarize_comparisons" dataset contains examples with:
 *   `prompt`: The original text to be summarized.
 *   `chosen`: A preferred summary of the prompt.
 *   `rejected`: A less preferred summary of the prompt.

We need to format and tokenize this dataset for DPO training. We'll use the GPT-2 tokenizer again and prepare the dataset in the format expected by `DPOTrainer`

In [ ]:
def format_dpo_dataset(examples):
    return {
        "prompt": examples["prompt"],
        "chosen": examples["chosen"],
        "rejected": examples["rejected"],
    }

formatted_dpo_dataset = dpo_dataset.map(format_dpo_dataset)

def tokenize_dpo_dataset(examples):
    tokenized_prompts = tokenizer(examples["prompt"], truncation=True, padding="longest")
    tokenized_chosen = tokenizer(examples["chosen"], truncation=True, padding="longest")
    tokenized_rejected = tokenizer(examples["rejected"], truncation=True, padding="longest")
    return {
        "prompt_input_ids": tokenized_prompts["input_ids"],
        "prompt_attention_mask": tokenized_prompts["attention_mask"],
        "chosen_input_ids": tokenized_chosen["input_ids"],
        "chosen_attention_mask": tokenized_chosen["attention_mask"],
        "rejected_input_ids": tokenized_rejected["input_ids"],
        "rejected_attention_mask": tokenized_rejected["attention_mask"],
    }

tokenized_dpo_dataset = formatted_dpo_dataset.map(tokenize_dpo_dataset, batched=True)
print(tokenized_dpo_dataset[0]) # Inspect a tokenized DPO example

Now we will set up the `DPOTrainer` from the `trl` library. We'll use GPT-2 as our base model for DPO fine-tuning. For DPO, we need a causal language model (for text generation).


In [ ]:
model_name_dpo = "distilgpt2"
ref_model_name_dpo = "distilgpt2" # Using the same model as reference for simplicity

# Load the causal language model (for text generation)
model_dpo = AutoModelForCausalLM.from_pretrained(model_name_dpo)
ref_model_dpo = AutoModelForCausalLM.from_pretrained(ref_model_name_dpo) # Reference model for DPO

dpo_training_args = TrainingArguments(
    output_dir="./dpo-distilgpt2-model",
    learning_rate=1e-5, # DPO often benefits from a smaller learning rate
    per_device_train_batch_size=2, # Reduce batch size if memory issues occur
    num_train_epochs=1, # For demonstration, we'll use a small number of epochs
    logging_steps=100,
    save_strategy="epoch",
    evaluation_strategy="no", # Evaluation in DPO is less straightforward, skipping for simplicity
)

dpo_trainer = DPOTrainer(
    model=model_dpo,
    ref_model=ref_model_dpo, # Reference model
    args=dpo_training_args,
    train_dataset=tokenized_dpo_dataset,
    tokenizer=tokenizer,
    beta=0.1, # Beta parameter controls the strength of preference modeling
)

dpo_trainer.train() # Start DPO training

After training, let's generate text using the DPO fine-tuned model and see if it produces summaries that are more aligned with the preferences it learned.

In [ ]:
def generate_dpo_text(prompt_text):
    inputs = tokenizer(prompt_text, return_tensors="pt").to(model_dpo.device)
    outputs = model_dpo.generate(**inputs, max_length=100, num_return_sequences=2)
    generated_texts = [tokenizer.decode(output, skip_special_tokens=True) for output in outputs]
    return generated_texts

prompt_for_dpo = "Summarize the following article about the benefits of exercise:" # A summarization prompt

print("\nGenerated Summaries from DPO Fine-tuned GPT-2:")
summaries = generate_dpo_text(prompt_for_dpo)
for summary in summaries:
    print(f"- {summary}")

TODO: To see the effect of DPO, you could compare these summaries with summaries generated by the original, pre-DPO fine-tuned GPT-2 model (from section 1) for the same prompt. Ideally, the DPO fine-tuned model's summaries should be more aligned with human-like preferences for summarization, as learned from the comparison dataset.

* Try different prompts for text generation with the base GPT-2 model and the DPO fine-tuned model. Observe how the outputs change.
* Try fine-tuning other models on other datasets for different tasks, such as text classification, question answering, or text summarization. You can find many datasets on the Hugging Face Hub ([https://huggingface.co/datasets](https://huggingface.co/datasets)).
* Experiment with other other LLM variants or different LLMs available on the Hugging Face Hub. Compare their performance and characteristics.
* Modify the training hyperparameters (learning rate, batch size, number of epochs, weight decay, beta in DPO) and observe how they affect the fine-tuning process and model performance.
* Investigate Parameter-Efficient Fine-tuning techniques like LoRA (Low-Rank Adaptation) to fine-tune large models more efficiently, especially when resources are limited. Hugging Face `peft` library (already installed) provides tools for this.
* Explore more advanced RLHF techniques and the theoretical underpinnings of preference learning and alignment.